# 中间件

控制和自定义代理执行的每一步

中间件提供了一种更严格地控​​制代理内部运行方式的方法。
核心代理循环包括调用模型，让模型选择要执行的工具，然后在不再需要调用任何工具时结束：

<img src="https://i-blog.csdnimg.cn/direct/96c3cd644cab4aad9985bfc3090ac813.png">

中间件在每个步骤之前和之后都会暴露钩子：

<img src="https://i-blog.csdnimg.cn/direct/1846606dfaaa4d5b8299423acce7d136.png">

## 中间件可以做什么？
- 监视器
通过日志记录、分析和调试来跟踪代理行为

- 调整
转换提示、工具选择和输出格式

- 控制
添加重试、回退和提前终止逻辑

- 执行
应用速率限制、防护措施和个人身份信息检测

## 内置中间件
LangChain 为常见用例提供预构建的中间件：
​
### 总结
当接近会话次数上限时，自动汇总对话历史记录。



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware


agent = create_agent(
    model="gpt-4o",
    tools=[weather_tool, calculator_tool],
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            max_tokens_before_summary=4000,  # Trigger summarization at 4000 tokens
            messages_to_keep=20,  # Keep last 20 messages after summary
            summary_prompt="Custom prompt for summarization...",  # Optional
        ),
    ],
)

### 人机交互
在工具调用执行之前，暂停代理执行，以便人工审批、编辑或拒绝工具

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


agent = create_agent(
    model="gpt-4o",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # Require approval, editing, or rejection for sending emails
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                # Auto-approve reading emails
                "read_email_tool": False,
            }
        ),
    ],
)

#### 配置选项
- interrupt_on dict  required

将工具名称映射到审批配置。值可以是 True（使用默认配置中断）、False（自动批准）或 InterruptOnConfig 对象。

- description_prefix string  default:"Tool execution requires approval"
行动请求说明的前缀

- InterruptOnConfig options:
    - allowed_decisions  允许的决定列表："approve"，"edit"或"reject"
    - description string | callable 用于自定义描述的静态字符串或可调用函数





### 人类提示缓存
使用 Anthropic 模型缓存重复的提示前缀，从而降低成本。

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_anthropic.middleware import AnthropicPromptCachingMiddleware
from langchain.agents import create_agent


LONG_PROMPT = """
Please be a helpful assistant.

<Lots more context ...>
"""

agent = create_agent(
    model=ChatAnthropic(model="claude-sonnet-4-5-20250929"),
    system_prompt=LONG_PROMPT,
    middleware=[AnthropicPromptCachingMiddleware(ttl="5m")],
)

# cache store
agent.invoke({"messages": [HumanMessage("Hi, my name is Bob")]})

# cache hit, system prompt is cached
agent.invoke({"messages": [HumanMessage("What's my name?")]})

- type stringdefault:"ephemeral"
缓存类型。"ephemeral"目前仅支持此类型。

- ttl stringdefault:"5m"
缓存内容的生存时间。有效值："5m"或"1h"

- min_messages_to_cache numberdefault:"0"
开始缓存前的最小消息数

- unsupported_model_behavior stringdefault:"warn"
使用非人格模型时的行为。选项："ignore"，，"warn"或"raise"

### 工具调用限制
通过限制工具调用次数来控制代理执行，可以全局限制所有工具的调用次数，也可以限制特定工具的调用次数。

要全局限制所有刀具或特定刀具的刀具调用次数，请进行设置`tool_name`。对于每个限制，请指定以下一项或两项：

- 线程限制（thread_limit）- 会话中所有运行的最大调用次数。此限制在后续调用中保持不变。需要检查点。
- 运行限制（run_limit）- 单次调用的最大次数。每回合重置。

退出行为：

| 行为 | 影响 | 最适合 |
| :--- | :--- | :--- |
| "continue" (default) | 拦截器调用次数超出预期并出现错误信息，代理继续执行操作 | 大多数使用场景下，代理都能优雅地处理限制。 |
| "error" | 立即引发异常 | 复杂的流程中，您需要手动处理限制错误。 |
| "end" | 停止 ToolMessage + AI 消息 | 单工具场景（如有其他工具待处理则出错） |


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware

# Global limit: max 20 calls per thread, 10 per run
global_limiter = ToolCallLimitMiddleware(
    thread_limit=20,
    run_limit=10,
)

# Tool-specific limit with default "continue" behavior
search_limiter = ToolCallLimitMiddleware(
    tool_name="search",
    thread_limit=5,
    run_limit=3,
)

# Thread limit only (no per-run limit)
database_limiter = ToolCallLimitMiddleware(
    tool_name="query_database",
    thread_limit=10,
)

# Strict enforcement with "error" behavior
web_scraper_limiter = ToolCallLimitMiddleware(
    tool_name="scrape_webpage",
    run_limit=2,
    exit_behavior="error",
)

# Immediate termination with "end" behavior
critical_tool_limiter = ToolCallLimitMiddleware(
    tool_name="delete_records",
    run_limit=1,
    exit_behavior="end",
)

# Use multiple limiters together
agent = create_agent(
    model="gpt-4o",
    tools=[search_tool, database_tool, scraper_tool],
    middleware=[
        global_limiter,
        search_limiter,
        database_limiter,
        web_scraper_limiter
    ],
)

### 模型回退
当主模型失效时，自动回退到备用模型。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware


agent = create_agent(
    model="gpt-4o",  # Primary model
    tools=[...],
    middleware=[
        ModelFallbackMiddleware(
            "gpt-4o-mini",  # Try first on error
            "claude-3-5-sonnet-20241022",  # Then this
        ),
    ],
)

### PII 检测
检测和处理对话中的个人身份信息。

非常适合：
- 医疗保健和金融应用需符合合规要求
- 需要清理日志的客服人员
- 任何处理敏感用户数据的应用程序




In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware


agent = create_agent(
    model="gpt-4o",
    tools=[...],
    middleware=[
        # Redact emails in user input
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        # Mask credit cards (show last 4 digits)
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        # Custom PII type with regex
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",  # Raise error if detected
        ),
    ],
)

- pii_type  string  required
要检测的 PII 类型。 可以是内置类型（email、credit_card、ip、ma​​c_address、url）或自定义类型名称。

-  strategy string  default:"redact"
    如何处理检测到的个人身份信息？选项：
    - "block"- 检测到异常时抛出异常
    - "redact"- 替换为[REDACTED_TYPE]
    - "mask"- 部分遮盖（例如，****-****-****-1234）
    - "hash"- 替换为确定性哈希

- detector function | regex
自定义检测器函数或正则表达式模式。如果未提供，则使用内置的 PII 类型检测器。

- apply_to_input booleandefault:"True"
在模型调用之前检查用户消息

- apply_to_output booleandefault:"False"
模型调用后检查 AI 消息

-  apply_to_tool_results boolean default:"False"
执行后检查工具结果消息





### 待办事项清单
为代理人配备任务规划和跟踪功能，以处理复杂的多步骤任务。

就像人类在记录和跟踪任务时效率更高一样，智能体也可以通过结构化的任务管理来分解复杂的问题，随着新信息的出现调整计划，并提高工作流程的透明度。

你可能已经注意到 Claude Code 中有类似的模式，它会在处理复杂的多部分任务之前列出待办事项清单。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool


@tool
def read_file(file_path: str) -> str:
    """Read contents of a file."""
    with open(file_path) as f:
        return f.read()


@tool
def write_file(file_path: str, content: str) -> str:
    """Write content to a file."""
    with open(file_path, 'w') as f:
        f.write(content)
    return f"Wrote {len(content)} characters to {file_path}"


@tool
def run_tests(test_path: str) -> str:
    """Run tests and return results."""
    # Simplified for example
    return "All tests passed!"


agent = create_agent(
    model="gpt-4o",
    tools=[read_file, write_file, run_tests],
    middleware=[TodoListMiddleware()],
)

result = agent.invoke({
    "messages": [HumanMessage("Refactor the authentication module to use async/await and ensure all tests pass")]
})

# The agent will use write_todos to plan and track:
# 1. Read current authentication module code
# 2. Identify functions that need async conversion
# 3. Refactor functions to async/await
# 4. Update function calls throughout codebase
# 5. Run tests and fix any failures

print(result["todos"])  # Track the agent's progress through each step

- system_prompt  string
自定义系统提示，用于指导待办事项的使用。如果未指定，则使用内置提示。
​
- tool_description string
工具的自定义描述write_todos。如果未指定，则使用内置描述。

### LLM工具选择器
使用 LLM 在调用主模型之前智能地选择相关工具。



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware


agent = create_agent(
    model="gpt-4o",
    tools=[tool1, tool2, tool3, tool4, tool5, ...],  # Many tools
    middleware=[
        LLMToolSelectorMiddleware(
            model="gpt-4o-mini",  # Use cheaper model for selection
            max_tools=3,  # Limit to 3 most relevant tools
            always_include=["search"],  # Always include certain tools
        ),
    ],
)

- model string | BaseChatModel
工具选择模型。 可以是模型字符串或 BaseChatModel 实例。 默认为代理的主要模型。
​
- system_prompt string
选择模型的说明。如果未指定，则使用内置提示。
​
- max_tools number
可选择的工具最大数量。默认无限制。
​
- always_include list[string]
选择中始终包含的工具名称列表

### 工具重试
使用可配置的指数退避算法自动重试失败的工具调用。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware


agent = create_agent(
    model="gpt-4o",
    tools=[search_tool, database_tool],
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,  # Retry up to 3 times
            backoff_factor=2.0,  # Exponential backoff multiplier
            initial_delay=1.0,  # Start with 1 second delay
            max_delay=60.0,  # Cap delays at 60 seconds
            jitter=True,  # Add random jitter to avoid thundering herd
        ),
    ],
)

- max_retries numberdefault:"2"
首次调用后的最大重试次数（默认值为 3 次）
​
- tools list[BaseTool | str]
可选的工具列表或工具名称，用于指定要应用重试逻辑的工具。如果指定None，则应用于所有工具。
​
- retry_on tuple[type[Exception], ...] | callabledefault:"(Exception,)"
可以是要重试的异常类型元组，也可以是接受异常并返回True是否应该重试的可调用对象。
​
- on_failure string | callabledefault:"return_message"
    当所有重试次数都用尽时的行为。选项：
    - "return_message" - 返回包含错误详情的工具消息（允许 LLM 处理故障）
    - "raise" - 重新引发异常（停止代理执行）
    - Custom callable - 自定义可调用对象 - 该函数接收异常并返回 ToolMessage 内容的字符串。
​
- backoff_factor numberdefault:"2.0"
乘数因子，用于指数退避。每个重试等待 initial_delay * (backoff_factor ** retry_number) 秒。设置为 0.0 则为恒定延迟。
​
- initial_delay numberdefault:"1.0"
指数退避的乘数。每次重试等待initial_delay * (backoff_factor ** retry_number)秒数。设置为 0.0 表示恒定延迟。
​
- initial_delay numberdefault:"1.0"
首次重试前的初始延迟时间（秒）
​
- max_delay numberdefault:"60.0"
重试之间的最大延迟时间（以秒为单位）（限制指数级退避增长）
​
- jitter booleandefault:"true"
是否在延迟中加入随机抖动（±25%）以避免群体雷鸣效应

## LLM 工具模拟器
使用 LLM 模拟工具执行以进行测试，用 AI 生成的响应替换实际的工具调用。




In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator


agent = create_agent(
    model="gpt-4o",
    tools=[get_weather, search_database, send_email],
    middleware=[
        # Emulate all tools by default
        LLMToolEmulator(),

        # Or emulate specific tools
        # LLMToolEmulator(tools=["get_weather", "search_database"]),

        # Or use a custom model for emulation
        # LLMToolEmulator(model="claude-sonnet-4-5-20250929"),
    ],
)

### 上下文编辑
通过精简、总结或清除工具使用情况来管理对话上下文。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit


agent = create_agent(
    model="gpt-4o",
    tools=[...],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(trigger=1000),  # Clear old tool uses
            ],
        ),
    ],
)

- edits list[ContextEdit]default:"[ClearToolUsesEdit()]"
ContextEdit可应用策略列表

- token_count_method stringdefault:"approximate"
令牌计数方法。选项："approximate"或"model"


ClearToolUsesEdit options:
​
- trigger numberdefault:"100000"
触发编辑的令牌计数
​
- clear_at_least numberdefault:"0"
可回收的最低token数量
​
- keep numberdefault:"3"
保留近期工具结果的数量
​
- clear_tool_inputs booleandefault:"False"
是否清除工具调用参数
​
- exclude_tools list[string]default:"()"
要从清除操作中排除的工具名称列表
​
- placeholder stringdefault:"[cleared]"
已清除输出的占位符文本




## 自定义中间件
通过在代理执行流程的特定点运行钩子来构建自定义中间件。

创建中间件有两种方法：
- 基于装饰器的- 适用于单钩中间件的快速简便
- 基于类的——对于具有多个钩子的复杂中间件来说功能更强大。


### 基于装饰器的中间件
对于只需要单个钩子的简单中间件，装饰器是添加功能的最快捷方式：

In [ ]:
from langchain.agents.middleware import before_model, after_model, wrap_model_call
from langchain.agents.middleware import AgentState, ModelRequest, ModelResponse, dynamic_prompt
from langchain.messages import AIMessage
from langchain.agents import create_agent
from langgraph.runtime import Runtime
from typing import Any, Callable


# Node-style: logging before model calls
@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"About to call model with {len(state['messages'])} messages")
    return None

# Node-style: validation after model calls
@after_model(can_jump_to=["end"])
def validate_output(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    last_message = state["messages"][-1]
    if "BLOCKED" in last_message.content:
        return {
            "messages": [AIMessage("I cannot respond to that request.")],
            "jump_to": "end"
        }
    return None

# Wrap-style: retry logic
@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as e:
            if attempt == 2:
                raise
            print(f"Retry {attempt + 1}/3 after error: {e}")

# Wrap-style: dynamic prompts
@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    user_id = request.runtime.context.get("user_id", "guest")
    return f"You are a helpful assistant for user {user_id}. Be concise and friendly."

# Use decorators in agent
agent = create_agent(
    model="gpt-4o",
    middleware=[log_before_model, validate_output, retry_model, personalized_prompt],
    tools=[...],
)

### 可用的装饰师
Node 风格（在特定执行点运行）：
- @before_agent- 在代理启动之前（每次调用一次）
- @before_model- 在每次模型调用之前
- @after_model- 每次模型响应后
- @after_agent- 代理完成后（每次调用一次）

包装式（拦截和控制执行）：
- @wrap_model_call- 每次模型调用前后
- @wrap_tool_call- 每次工具调用前后

便利装饰器：
- @dynamic_prompt- 生成动态系统提示（相当于@wrap_model_call修改提示）
​


### 基于类的中间件
​
#### 两种钩子样式

Node 风格的钩子
在执行流程中的特定点运行：
- before_agent- 在代理启动之前（每次调用一次）
- before_model- 在每次模型调用之前
- after_model- 每次模型响应后
- after_agent- 代理完成后（每次调用最多一次）



示例：日志中间件

In [ ]:
from langchain.agents.middleware import AgentMiddleware, AgentState
from langgraph.runtime import Runtime
from typing import Any

class LoggingMiddleware(AgentMiddleware):
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"About to call model with {len(state['messages'])} messages")
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"Model returned: {state['messages'][-1].content}")
        return None

例如：对话时长限制

In [ ]:
from langchain.agents.middleware import AgentMiddleware, AgentState
from langchain.messages import AIMessage
from langgraph.runtime import Runtime
from typing import Any

class MessageLimitMiddleware(AgentMiddleware):
    def __init__(self, max_messages: int = 50):
        super().__init__()
        self.max_messages = max_messages

    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if len(state["messages"]) == self.max_messages:
            return {
                "messages": [AIMessage("Conversation limit reached.")],
                "jump_to": "end"
            }
        return None

缠绕式钩子

在调用处理程序时拦截执行并进行控制：
- wrap_model_call- 每次模型调用前后
- wrap_tool_call- 每次工具调用前后

您可以决定处理程序是被调用零次（短路）、一次（正常流程）还是多次（重试逻辑）。

示例：模型重试中间件

In [ ]:
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from typing import Callable

class RetryMiddleware(AgentMiddleware):
    def __init__(self, max_retries: int = 3):
        super().__init__()
        self.max_retries = max_retries

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        for attempt in range(self.max_retries):
            try:
                return handler(request)
            except Exception as e:
                if attempt == self.max_retries - 1:
                    raise
                print(f"Retry {attempt + 1}/{self.max_retries} after error: {e}")

示例：动态模型选择

In [ ]:
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

class DynamicModelMiddleware(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        # Use different model based on conversation length
        if len(request.messages) > 10:
            request.model = init_chat_model("gpt-4o")
        else:
            request.model = init_chat_model("gpt-4o-mini")

        return handler(request)

示例：工具调用监控

In [ ]:
from langchain.tools.tool_node import ToolCallRequest
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import ToolMessage
from langgraph.types import Command
from typing import Callable

class ToolMonitoringMiddleware(AgentMiddleware):
    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command],
    ) -> ToolMessage | Command:
        print(f"Executing tool: {request.tool_call['name']}")
        print(f"Arguments: {request.tool_call['args']}")

        try:
            result = handler(request)
            print(f"Tool completed successfully")
            return result
        except Exception as e:
            print(f"Tool failed: {e}")
            raise

#### 自定义状态模式
中间件可以使用自定义属性扩展代理的状态。定义一个自定义状态类型并将其设置为state_schema：

In [ ]:
from langchain.agents.middleware import AgentState, AgentMiddleware
from typing_extensions import NotRequired
from typing import Any

class CustomState(AgentState):
    model_call_count: NotRequired[int]
    user_id: NotRequired[str]

class CallCounterMiddleware(AgentMiddleware[CustomState]):
    state_schema = CustomState

    def before_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
        # Access custom state properties
        count = state.get("model_call_count", 0)

        if count > 10:
            return {"jump_to": "end"}

        return None

    def after_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
        # Update custom state
        return {"model_call_count": state.get("model_call_count", 0) + 1}
    
agent = create_agent(
    model="gpt-4o",
    middleware=[CallCounterMiddleware()],
    tools=[...],
)

# Invoke with custom state
result = agent.invoke({
    "messages": [HumanMessage("Hello")],
    "model_call_count": 0,
    "user_id": "user-123",
})

#### 执行令
使用多个中间件时，了解执行顺序至关重要：


In [ ]:
agent = create_agent(
    model="gpt-4o",
    middleware=[middleware1, middleware2, middleware3],
    tools=[...],
)

##### 执行流程（点击展开）

- 钩子按顺​​序运行之前：
    - middleware1.before_agent()
    - middleware2.before_agent()
    - middleware3.before_agent()
- 代理循环开始：
    - middleware1.before_model()
    - middleware2.before_model()
    - middleware3.before_model()
- 像函数调用一样嵌套包装钩子
    - middleware1.wrap_model_call()
    - middleware2.wrap_model_call()
    - middleware3.wrap_model_call()
    - 模型
- 钩子按相反顺序运行后：
    - middleware3.after_model()
    - middleware2.after_model()
    - middleware1.after_model()
- 代理循环结束：
- 代理循环结束：
    - middleware1.after_agent()
    - middleware2.after_agent()
    - middleware3.after_agent()


##### 关键规则：
- before_*钩子：从第一个到最后一个
- after_*钩子：从后到前（反向）
- wrap_*hooks：嵌套式（第一个中间件包裹所有其他中间件）

#### 特工跳跃
要提前退出中间件，请返回一个包含以下内容的字典jump_to：

In [ ]:
class EarlyExitMiddleware(AgentMiddleware):
    def before_model(self, state: AgentState, runtime) -> dict[str, Any] | None:
        # Check some condition
        if should_exit(state):
            return {
                "messages": [AIMessage("Exiting early due to condition.")],
                "jump_to": "end"
            }
        return None

##### 可跳跃目标：
- "end"跳转到代理执行的末尾
- "tools"跳转到工具节点
- "model"跳转到模型节点（或第一个before_model钩子）

> 重要：从 before_model 或 after_model 跳转时，跳转到 "model "会导致所有 before_model 中间件重新运行。

要启用跳转，请使用 @hook_config(can_jump_to=[...]) 来装饰您的钩子：

In [ ]:
from langchain.agents.middleware import AgentMiddleware, hook_config
from typing import Any

class ConditionalMiddleware(AgentMiddleware):
    @hook_config(can_jump_to=["end", "tools"])
    def after_model(self, state: AgentState, runtime) -> dict[str, Any] | None:
        if some_condition(state):
            return {"jump_to": "end"}
        return None

## 最佳实践
- 保持中间件的专注——每个中间件都应该做好一件事。
- 优雅地处理错误——不要让中间件错误导致代理崩溃。
- 使用合适的钩子类型：
    - 用于顺序逻辑（日志记录、验证）的节点式编程
    - 控制流（重试、回退、缓存）的包装式
- 清楚地记录所有自定义状态属性
- 在集成之前，对中间件进行独立的单元测试。
- 考虑执行顺序——将关键中间件放在列表的最前面
- 尽可能使用内置中间件，不要重复造轮子 :)
​


## 示例
​
动态选择工具
在运行时选择合适的工具来提高性能和准确性。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, ModelRequest
from typing import Callable


class ToolSelectorMiddleware(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        """Middleware to select relevant tools based on state/context."""
        # Select a small, relevant subset of tools based on state/context
        relevant_tools = select_relevant_tools(request.state, request.runtime)
        request.tools = relevant_tools
        return handler(request)

agent = create_agent(
    model="gpt-4o",
    tools=all_tools,  # All available tools need to be registered upfront
    # Middleware can be used to select a smaller subset that's relevant for the given run.
    middleware=[ToolSelectorMiddleware()],
)

In [ ]:
from dataclasses import dataclass
from typing import Literal, Callable

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain_core.tools import tool


@tool
def github_create_issue(repo: str, title: str) -> dict:
    """Create an issue in a GitHub repository."""
    return {"url": f"https://github.com/{repo}/issues/1", "title": title}

@tool
def gitlab_create_issue(project: str, title: str) -> dict:
    """Create an issue in a GitLab project."""
    return {"url": f"https://gitlab.com/{project}/-/issues/1", "title": title}

all_tools = [github_create_issue, gitlab_create_issue]

@dataclass
class Context:
    provider: Literal["github", "gitlab"]

class ToolSelectorMiddleware(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        """Select tools based on the VCS provider."""
        provider = request.runtime.context.provider

        if provider == "gitlab":
            selected_tools = [t for t in request.tools if t.name == "gitlab_create_issue"]
        else:
            selected_tools = [t for t in request.tools if t.name == "github_create_issue"]

        request.tools = selected_tools
        return handler(request)

agent = create_agent(
    model="gpt-4o",
    tools=all_tools,
    middleware=[ToolSelectorMiddleware()],
    context_schema=Context,
)

# Invoke with GitHub context
agent.invoke(
    {
        "messages": [{"role": "user", "content": "Open an issue titled 'Bug: where are the cats' in the repository `its-a-cats-game`"}]
    },
    context=Context(provider="github"),
)